# Derivatives Pricing

**Quantitative Research Portfolio - Module 5**

**Author**: Kevin J.D. Metzler  
**Date**: May 4, 2026

This notebook demonstrates analytical and numerical pricing techniques implemented in the `05_derivatives_pricing` module:

1. **Black-Scholes** pricing, Greeks, and implied volatility
2. **Binomial Tree** pricing for European and American options
3. **Monte Carlo** pricing under GBM
4. **Black-76** pricing for options on forwards/futures
5. **Term Structure Models** (Vasicek and CIR) for zero-coupon bonds

## Mathematical Outline

**Black-Scholes (European Call)**

$$C = S_0 e^{-qT} N(d_1) - K e^{-rT} N(d_2)$$

$$d_1 = \frac{\ln(S_0/K) + (r - q + \tfrac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

**Binomial Tree (CRR)**

$$u = e^{\sigma\sqrt{\Delta t}}, \quad d = 1/u, \quad p = \frac{e^{(r-q)\Delta t} - d}{u - d}$$

**Monte Carlo (GBM)**

$$S_T = S_0 \exp\left((r-q-\tfrac{1}{2}\sigma^2)T + \sigma\sqrt{T}Z\right)$$

**Term Structure Models**

Vasicek and CIR models provide closed-form prices for zero-coupon bonds with mean-reverting short rates.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("."))

from derivatives_pricing import (
    OptionParams,
    BlackScholesPricer,
    BinomialTreePricer,
    MonteCarloPricer,
    VasicekModel,
    CIRModel,
    black_76_price,
    forward_price,
    put_call_parity,
)

sns.set_style("whitegrid")
%matplotlib inline

## 1. Black-Scholes Pricing and Greeks

In [ ]:
params = OptionParams(
    spot=100,
    strike=100,
    maturity=1.0,
    rate=0.05,
    volatility=0.2,
    dividend_yield=0.01,
    option_type="call",
)

bs = BlackScholesPricer(params)
price = bs.price()
greeks = bs.greeks()

summary = pd.DataFrame(
    [{"price": price, **greeks}]
)
summary

**How to read the table:**

- **price** is the model-implied fair value of the option.
- **delta** is the first-order sensitivity to spot (price change per $1 move in the underlying).
- **gamma** captures curvature: how quickly delta changes as spot moves.
- **vega** measures sensitivity to volatility (price change per 1.0 vol unit).
- **theta** is time decay (per year) holding all else fixed.
- **rho** is sensitivity to the risk-free rate.

In [ ]:
spots = np.linspace(60, 140, 60)
prices = []
deltas = []
for s in spots:
    p = OptionParams(
        spot=float(s),
        strike=100,
        maturity=1.0,
        rate=0.05,
        volatility=0.2,
        dividend_yield=0.01,
        option_type="call",
    )
    pricer = BlackScholesPricer(p)
    prices.append(pricer.price())
    deltas.append(pricer.greeks()["delta"])

fig, ax1 = plt.subplots()
ax1.plot(spots, prices, color="tab:blue", label="Price")
ax1.set_xlabel("Spot")
ax1.set_ylabel("Option Price", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(spots, deltas, color="tab:orange", linestyle="--", label="Delta")
ax2.set_ylabel("Delta", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

plt.title("Black-Scholes Call: Price and Delta vs Spot")
fig.tight_layout()
plt.show()

### Implied Volatility Example

In [ ]:
market_price = price * 1.05
implied_vol = BlackScholesPricer.implied_volatility(market_price, params)

pd.DataFrame(
    [{"market_price": market_price, "implied_vol": implied_vol}]
)

The implied volatility backs out the volatility that makes the Black-Scholes price match the observed market price. Because the market price is set 5% above the model price, the implied volatility is higher than the input volatility (0.20).

In [ ]:
vols = np.linspace(0.05, 0.6, 60)
vol_prices = []
for v in vols:
    p = OptionParams(
        spot=100,
        strike=100,
        maturity=1.0,
        rate=0.05,
        volatility=float(v),
        dividend_yield=0.01,
        option_type="call",
    )
    vol_prices.append(BlackScholesPricer(p).price())

plt.plot(vols, vol_prices, label="Model price")
plt.axvline(params.volatility, color="gray", linestyle=":", label="Input vol")
plt.axvline(implied_vol, color="tab:red", linestyle="--", label="Implied vol")
plt.xlabel("Volatility")
plt.ylabel("Option Price")
plt.title("Price vs Volatility (Implied Volatility)")
plt.legend()
plt.show()

### Put-Call Parity Check

In [ ]:
call_params = OptionParams(
    spot=100,
    strike=100,
    maturity=1.0,
    rate=0.05,
    volatility=0.2,
    dividend_yield=0.01,
    option_type="call",
)
put_params = OptionParams(
    spot=100,
    strike=100,
    maturity=1.0,
    rate=0.05,
    volatility=0.2,
    dividend_yield=0.01,
    option_type="put",
)

call_price = BlackScholesPricer(call_params).price()
put_price = BlackScholesPricer(put_params).price()
parity_residual = put_call_parity(call_params, call_price, put_price)

pd.DataFrame(
    [{"call_price": call_price, "put_price": put_price, "parity_residual": parity_residual}]
)

A parity residual close to zero indicates internal consistency between the call and put prices given the same inputs.

## 2. Binomial Tree: European vs American Put

In [ ]:
params_put = OptionParams(
    spot=50,
    strike=55,
    maturity=0.5,
    rate=0.03,
    volatility=0.25,
    dividend_yield=0.0,
    option_type="put",
)

steps = [25, 50, 100, 200, 400]
rows = []
for n in steps:
    euro = BinomialTreePricer(params_put, steps=n, is_american=False).price()
    amer = BinomialTreePricer(params_put, steps=n, is_american=True).price()
    rows.append({"steps": n, "european_put": euro, "american_put": amer})

binom_df = pd.DataFrame(rows)
binom_df

The American put should price at or above the European put because early exercise can add value. As the number of steps increases, the tree prices stabilize (converge).

In [ ]:
plt.plot(binom_df["steps"], binom_df["european_put"], marker="o", label="European Put")
plt.plot(binom_df["steps"], binom_df["american_put"], marker="s", label="American Put")
plt.xlabel("Tree Steps")
plt.ylabel("Option Price")
plt.title("Binomial Tree Convergence")
plt.legend()
plt.show()

## 3. Monte Carlo Pricing and Convergence

In [ ]:
mc_params = OptionParams(
    spot=100,
    strike=95,
    maturity=1.0,
    rate=0.04,
    volatility=0.2,
    dividend_yield=0.01,
    option_type="call",
)

bs_price = BlackScholesPricer(mc_params).price()
paths = [2000, 5000, 10000, 20000, 40000]
records = []
for n in paths:
    mc = MonteCarloPricer(mc_params, n_paths=n, n_steps=1, seed=42)
    price_mc, stderr = mc.price()
    records.append({"paths": n, "mc_price": price_mc, "stderr": stderr})

mc_df = pd.DataFrame(records)
mc_df

**Interpreting the table:** the Monte Carlo price should approach the Black-Scholes price as the number of paths increases. The standard error shrinks at roughly the rate $1/\sqrt{N}$, so larger simulations yield tighter confidence bands.

In [ ]:
plt.errorbar(
    mc_df["paths"],
    mc_df["mc_price"],
    yerr=1.96 * mc_df["stderr"],
    marker="o",
    capsize=3,
    label="Monte Carlo (95% CI)",
)
plt.axhline(bs_price, color="black", linestyle="--", label="Black-Scholes")
plt.xscale("log")
plt.xlabel("Number of Paths (log scale)")
plt.ylabel("Option Price")
plt.title("Monte Carlo Convergence")
plt.legend()
plt.show()

In [ ]:
mc_hist = MonteCarloPricer(mc_params, n_paths=20000, n_steps=1, seed=7)
terminal = mc_hist._simulate_terminal()
payoffs = np.maximum(terminal - mc_params.strike, 0.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(terminal, bins=40, color="steelblue", alpha=0.8)
axes[0].set_title("Simulated Terminal Prices")
axes[0].set_xlabel("$S_T$")
axes[0].set_ylabel("Frequency")

axes[1].hist(payoffs, bins=40, color="seagreen", alpha=0.8)
axes[1].set_title("Discounted Payoffs (Call)")
axes[1].set_xlabel("Payoff")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

## 4. Black-76 Pricing for Futures Options

In [ ]:
fwd = forward_price(spot=100, rate=0.05, dividend_yield=0.01, maturity=1.0)
black76_price = black_76_price(forward=fwd, strike=100, maturity=1.0, rate=0.05, volatility=0.2)

pd.DataFrame(
    [{"forward": fwd, "black76_price": black76_price}]
)

## 5. Term Structure Models (Vasicek and CIR)

In [ ]:
maturities = np.array([0.5, 1, 2, 3, 5, 7, 10])

vasicek = VasicekModel(a=0.5, b=0.03, sigma=0.01, r0=0.02)
cir = CIRModel(a=0.6, b=0.04, sigma=0.08, r0=0.03)

vasicek_prices = np.array([vasicek.zero_coupon_bond_price(t) for t in maturities])
cir_prices = np.array([cir.zero_coupon_bond_price(t) for t in maturities])

vasicek_yields = -np.log(vasicek_prices) / maturities
cir_yields = -np.log(cir_prices) / maturities

term_df = pd.DataFrame(
    {
        "maturity": maturities,
        "vasicek_price": vasicek_prices,
        "cir_price": cir_prices,
        "vasicek_yield": vasicek_yields,
        "cir_yield": cir_yields,
    }
)
term_df

In [ ]:
plt.plot(maturities, vasicek_yields, marker="o", label="Vasicek")
plt.plot(maturities, cir_yields, marker="s", label="CIR")
plt.xlabel("Maturity (years)")
plt.ylabel("Zero Rate")
plt.title("Term Structure Comparison")
plt.legend()
plt.show()